<h1> Implementing Random Forest Classifier </h1>
<h3> Importing Required Libraries </h3>

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
%matplotlib inline
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV
from joblib import dump, load #to save ur model
from sklearn.model_selection import GridSearchCV

<h3> Reading the Data from CSV file </h3>
<h4>1. Accident Data
<br>2. Casualities Data
<br>3. Vehicles Data
<br> We will try to add more and more features this time to build a highly accurate classifier.
<br><h3>Provided:</h3> <h5>Previously we have only used Accident Data to build our classifier without balancing the data and we didn't got a single considerable classifier which can classify most effectively in all of the accident_severity(3,2,1) scenarios.</h5></h4>

In [ ]:
df_Acc=pd.read_csv('/Users/manasc12/MyProjects/Data/RealUKAccidentAnalysisData/dftRoadSafetyData_Accidents_2018.csv')
df_Veh=pd.read_csv('/Users/manasc12/MyProjects/Data/RealUKAccidentAnalysisData/dftRoadSafetyData_Vehicles_2018.csv')
df_Acc.head(5)

In [ ]:
df_Veh

<h3> Data Cleaning and Wrangling

In [ ]:
#Cleaning up the null valued rows
df_Acc_clnd=df_Acc[~df_Acc.isnull().any(axis=1)]
df_Veh_clnd=df_Veh[~df_Veh.isnull().any(axis=1)]

In [ ]:
#Transforming 'Time' Attribute from hh:mm to only minutes
a= np.array([(int(str(x).split(':')[0])*60+int(str(x).split(':')[1])) for x in df_Acc_clnd['Time'].values])
df_Acc_clnd['Time_Conv']=a

In [ ]:
#Visualizing the shape 
print('df_Acc.shape---'+str(df_Acc.shape))
print('df_Veh.shape---'+str(df_Veh.shape))
print('df_Acc_clnd.shape---'+str(df_Acc_clnd.shape))
print('df_Veh_clnd.shape---'+str(df_Veh_clnd.shape))
print('----------------------------------------------------------------------')
print('df_Acc_clnd unique values---')
print('----------------------------------------------------------------------')
print(df_Acc_clnd.nunique(axis=0))
print('----------------------------------------------------------------------')
print('df_Veh_clnd unique values---')
print('----------------------------------------------------------------------')
print(df_Veh_clnd.nunique(axis=0))

In [ ]:
#and Data Types
print('----------------------------------------------------------------------')
print('df_Acc_clnd Data types---')
print('----------------------------------------------------------------------')
print(df_Acc_clnd.dtypes)
print('----------------------------------------------------------------------')
print('df_Veh_clnd Data types---')
print('----------------------------------------------------------------------')
print(df_Veh_clnd.dtypes)

<h4>There are 2 sheets(2 dataframes). So lets check which are the columns(attributes/features) on which we can perform our join operation. Inorder to get a single dataframe.

In [ ]:
print('-----------------------------------------')
print('Columns same in Accident and Vehicle:')
print('-----------------------------------------')
for c1 in df_Acc_clnd.columns:
    for c2 in df_Veh_clnd.columns:
        if(c1==c2):
            print(c1)

<h4>Now that we know our join keys so lets merge our 3 dataframes

In [ ]:
df_All_clnd=pd.merge(df_Acc_clnd,df_Veh_clnd,how='inner',on=['Accident_Index'])

<h4> Bifurcating Attributes for Training and Target

In [ ]:
df=df_All_clnd.reset_index(drop=True)

##Drop the columns which are irrelevant to get before the accident or totally irrelevant w.r.t the prediction model
df.drop(['Date','Time','Accident_Index','Number_of_Casualties','Vehicle_Reference'
         ,'Police_Force','Did_Police_Officer_Attend_Scene_of_Accident','1st_Point_of_Impact'
         ,'Skidding_and_Overturning','Vehicle_Leaving_Carriageway','Hit_Object_off_Carriageway'
         ,'Driver_Home_Area_Type','Carriageway_Hazards','Towing_and_Articulation','Vehicle_Location-Restricted_Lane'
        ,'Hit_Object_in_Carriageway'],axis=1,inplace=True)

target_c=['Accident_Severity']
df_train=df.drop(target_c,axis=1)
df_target=df[target_c]

<h4> Balancing/Sampling

In [ ]:
#Determining Balance of our Data Samples
acc_sev_counts=pd.value_counts(df_target['Accident_Severity'],sort=True)
plt.figure(figsize=(5,5))
acc_sev_counts.plot(kind='bar')
plt.xlabel('Accident Severity')
plt.ylabel('No. Of Records')


<h3>As u can see our data set is highly unbalanced w.r.t our output feature.
    <h5><br>1. And probably the reason for which we aren't able to train an effective model in our previous attempts.
    <br>2. But now we will resample our data to balance it and then we will apply our RandomForestClassifier on top of it.
    <br>3. But before applying resampling we would like to do some Label-Encoding as the classifier doesn't accept string data types.

<h4> LabelEncoding

In [ ]:
labelEncodeCols=['LSOA_of_Accident_Location','Local_Authority_(Highway)']
#labelEncodeCols=['Local_Authority_(Highway)']
Local_Authority_Highway               = LabelEncoder()
LSOA_of_Accident_Location               = LabelEncoder()

Local_Authority_Highway.fit(df_train['Local_Authority_(Highway)'])
LSOA_of_Accident_Location.fit(df_train['LSOA_of_Accident_Location'])

df_train['Local_Authority_(Highway)_E']=Local_Authority_Highway.transform(df_train['Local_Authority_(Highway)'])
df_train['LSOA_of_Accident_Location_E']=LSOA_of_Accident_Location.transform(df_train['LSOA_of_Accident_Location'])
df_train.drop(labelEncodeCols,axis=1,inplace=True)


In [ ]:
#Performing Over Sampling
#There are various sampling technique you can use like;- NearMiss, SMOTETomek, RandomOverSampler
## RandomOverSampler to handle imbalanced data

os =  RandomOverSampler(random_state=42)
df_train_smpl, df_target_smpl = os.fit_sample(df_train, df_target)

In [ ]:
#Checking the new sampled Data set for balance
acc_sev_counts=pd.value_counts(df_target_smpl['Accident_Severity'],sort=True)
plt.figure(figsize=(5,5))
acc_sev_counts.plot(kind='bar')
plt.xlabel('Accident Severity')
plt.ylabel('No. Of Records')


<h3> Splitting up Training and Testing Data Samples

In [ ]:
x1_train, x1_test, y1_train, y1_test = train_test_split(df_train_smpl,df_target_smpl.to_numpy(),test_size=0.7,random_state=42)
y1_train=y1_train.reshape(len(y1_train),)
y1_test=y1_test.reshape(len(y1_test),)

<h3> Implementing Random Forest Classifier With More features Than before

<h4>1. Model with LabelEncoded Data(Sampled)+ All features(63)

In [ ]:
model_W_Le = RandomForestClassifier(n_estimators=50)
model_W_Le.fit(x1_train, y1_train)
print("{}".format(model_W_Le.score(x1_test, y1_test)))
metrics.confusion_matrix(y1_test, model_W_Le.predict(x1_test), labels=[3, 2, 1])

<h4> Getting The Feature Importance
<h5> The Code below is only Usable if you have Data Frames( i.e. x1_train , y1_train ) as Traning and ( i.e. x1_test , y1_test ) as Testing Data.

In [ ]:
# Get numerical feature importances
importances = list(model_W_Le.feature_importances_)
# List of tuples with variable and importance
feature_importances = [(feature, round(importance, 2)) for feature, importance in zip(x1_train.columns, importances)]
# Sort the feature importances by most important first
feature_importances = sorted(feature_importances, key = lambda x: x[1], reverse = True)
# Print out the feature and importances 
[print('Feature--------{}: Importance: {}'.format(*pair)) for pair in feature_importances];

<h3> Visualize a random Tree from the Forest

In [ ]:

# Import tools needed for visualization
from sklearn.tree import export_graphviz
import pydot

# Pull out one tree from the forest
tree = model_W_Le.estimators_[5]

# Export the image to a dot file
export_graphviz(tree, out_file = 'tree.dot', feature_names = x1_train.columns, rounded = True, precision = 1)





In [ ]:
# Use dot file to create a graph
(graph, ) = pydot.graph_from_dot_file('tree.dot')

In [ ]:
# Write graph to a png file
graph.write_png('tree.jpg')

<h3>Now Lets Implement Our Classifier with Only the most Important features</h3>
<h4><br>2. Model with LabelEncoded Data(Sampled) + Most Important Features(Found above)</h4>
<h7>-----------------------------------------------------------------------------------------------------------------------------------------------------------------</h7>
<h7><br>Here I have considered top features according to their feature_importance>= 2%.

In [ ]:
#Only keeping the most important Features
most_imp_ftr_cols=np.array(feature_importances)[np.array(feature_importances)[:,1].astype(float)>=0.02][:,0]
most_imp_ftr_cols

In [ ]:
x2_train=x1_train[most_imp_ftr_cols]
x2_test=x1_test[most_imp_ftr_cols]
y2_train=y1_train
y2_test=y1_test

In [ ]:
#Re-training the same model
model_W_Le.fit(x2_train, y2_train)
print("{}".format(model_W_Le.score(x2_test, y2_test)))
metrics.confusion_matrix(y2_test, model_W_Le.predict(x2_test), labels=[3, 2, 1])

<h3> Now Lets Implement Our Classifier with some HyperParameters using Parameter Tuning</h3>
<h7>-----------------------------------------------------------------------------------------------------------------------------------------------------------------</h7>
<h7><br> Previously we have tried implementing Random Forest Classifier only with Accident Dataset. But the results didn't look promising for: <br>Accident Severity=2 <b>(Serious)</b> and Accident Severity=1 <b>(Fatal)</b>
<br>Now that we have: 
<br>1. Sampled our dataset w.r.t our output feature.
<br>And we've got =~ 89.5% accuracy with just selecting top Features. We may now apply some hyperparameter tuning and try to improve accuracy as much as possible. </h7>

<h4>3. Model with LabelEncoded Data(Sampled) + Most Important Features + HyperParameters + RandomizedSearchCV + CrossValidation 

In [ ]:
# Number of trees in random forest
n_estimators = [int(x) for x in np.linspace(start = 20, stop = 100, num = 10)]
# Number of features to consider at every split
max_features = ['auto', 'sqrt']
# Maximum number of levels in tree
max_depth = [int(x) for x in np.linspace(10, 110, num = 11)]
max_depth.append(None)
# Minimum number of samples required to split a node
min_samples_split = [2, 5, 10]
# Minimum number of samples required at each leaf node
min_samples_leaf = [1, 2, 4]
# Method of selecting samples for training each tree
bootstrap = [True, False]
# Create the random grid
random_grid = {'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf,
               'bootstrap': bootstrap}
print(random_grid)

In [ ]:
# Use the random grid to search for best hyperparameters
# First create the base model to tune
model_W_Le_Hp = RandomForestClassifier()
# Random search of parameters, using 3 fold cross validation, 
# search across 50 different combinations, and use all available cores
model_W_Le_Hp_Rnd = RandomizedSearchCV(estimator = model_W_Le_Hp, param_distributions = random_grid, n_iter = 50, cv = 3, verbose=2, random_state=42, n_jobs = -1)
# Fit the random search model
model_W_Le_Hp_Rnd.fit(x2_train, y2_train)


In [ ]:
dump(model_W_Le_Hp_Rnd, 'RF_model_W_Le_Hp_Rnd.joblib') 
#model_W_Le_Hp_Rnd=load('RF_model_W_Le_Hp_Rnd.joblib')

In [ ]:
#View the best parameters of the best estimator
model_W_Le_Hp_Rnd.best_params_

In [ ]:
#Test ur model with the best estimator 
metrics.accuracy_score(y2_test, model_W_Le_Hp_Rnd.best_estimator_.predict(x2_test))

In [ ]:
#Test ur model with the best estimator 
metrics.confusion_matrix(y2_test, model_W_Le_Hp_Rnd.best_estimator_.predict(x2_test), labels=[3, 2, 1])

<h4>4. Model with LabelEncoded Data(Sampled) + Most Important Features + HyperParameters + GridSearchCV + CrossValidation 

In [ ]:
# Create the parameter grid based on the results of ur above random search 
param_grid_2 = {
    'bootstrap': [False],
    'max_depth': [30, 40, 50],
    'max_features': ['auto'],
    'min_samples_leaf': [1, 2, 3],
    'min_samples_split': [2, 4, 8],
    'n_estimators': [63, 73, 83]
}

# Instantiate the grid search model
model_W_Le_Hp_Grd = GridSearchCV(estimator = model_W_Le_Hp, param_grid = param_grid_2, 
                          cv = 3, n_jobs = -1, verbose = 2)
model_W_Le_Hp_Grd.fit(x2_train, y2_train)

In [ ]:
dump(model_W_Le_Hp_Grd, 'RF_model_W_Le_Hp_Grd.joblib') 
#model_W_Le_Hp_Rnd=load('RF_model_W_Le_Hp_Grd.joblib')

In [ ]:
#View the best parameters of the best estimator
model_W_Le_Hp_Grd.best_params_

In [ ]:
#Test ur model with the best estimator 
best_Mdl_W_Le_Hp_Grd=model_W_Le_Hp_Grd.best_estimator_
best_Mdl_W_Le_Hp_Grd.score(x2_test, y2_test)

In [ ]:
metrics.confusion_matrix(y2_test,best_Mdl_W_Le_Hp_Grd.predict(x2_test),labels=[3,2,1])

<h3>Accuracy Description After Adding More Features + Performing Sampling:-
<br>-------------------------------------------------------------------------------------------------
<h4>1. Model with LabelEncoded Data(Sampled) + All features -------------------- 89.8%
<br>2. Model with LabelEncoded Data(Sampled) + Most Important Features ----- 89.4%
<br>3. Model with LabelEncoded Data(Sampled) + Most Important Features + HyperParameters + RandomizedSearchCV + CrossValidation -- 91.0%
<br>4. Model with LabelEncoded Data(Sampled) + Most Important Features + HyperParameters + GridSearchCV + CrossValidation ------------ 91.14%

From Confusion matrix calucalated in each of the above cases you can now verify that our newly Trained classifier can effectively classify in each of the 3 Accident Severity types.